### Importing Python packages

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
import rasterio
from rasterio import features
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import numpy as np
import rioxarray as rxr
import xarray as xr
import dask.array as da
import dask
import dask.dataframe as dd
import cupy as cp
import time
import logging
from joblib import load
from dask.diagnostics import ProgressBar
from scipy.spatial.distance import cdist
from sklearn.base import TransformerMixin
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
import gc
from scipy.spatial.distance import euclidean
from sklearn.metrics import pairwise_distances_argmin_min
from shapely.geometry import Point
import warnings
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
warnings.filterwarnings('ignore')

gc.disable()  # Disable automatic garbage collection

# Run garbage collection after processing large batch works
def free_memory():
    gc.collect()
    client.run(gc.collect)  

# Set seed
seed = 25

In [ ]:
import os
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TERMINATE"] = "0.95"  # Make workers to use up to 95% of memory limit. 
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__PAUSE"] = "0.85"   # Pause at 85% to prevent memory overload and crash. 
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TARGET"] = "0.8" # Keep memory usage around the target of 80% for optimal performance.

cluster = LocalCluster(n_workers=6, threads_per_worker=2, memory_limit='5GB')
client = Client(cluster)
print(f"Dashboard link: {client.dashboard_link}")

### Load Training Data

In [ ]:
# Load training data file
file = r"original_training_data.gpkg"

gdf = gpd.read_file(file, driver = "gpkg", geometry = 'geometry')

# Drop NA values from the dataset
gdf.dropna(inplace = True)
gdf.reset_index(drop=True, inplace=True)

print("Total number of data: ", gdf.shape)

# change the order of the variables in the dataset, similar to the raster data; delete the geometry column
training_columns =  ['soc', 'clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']
gdf = gdf[training_columns]

training_df = xr.Dataset(gdf)

### Functions for Loading raster data

#### Preprocessing raster data

In [ ]:
def preprocess_raster_data(file_path, band_names, chunk_size=(1000, 1000)):
    """
    Load and preprocess raster data with memory-efficient processing.
    Steps include renaming bands, handling missing data, recoding land-use types,
    and filtering out data having non-values. 
    
    Parameters:
        - file_path (str): The input raster file path.
        - band_names (list): List of band names.
        - chunk_size (tuple): Size of chunks. 
    
    Returns:
        - xarray.Dataset: The preprocessed raster dataset.
        - xarray.DataArray: The original raster data.
    """
    # Landuse recoding dictionary
    landuse_codes = {
        **{i: 'grassland' for i in range(10100, 12001, 100)},
        **{i: 'wetland' for i in range(20100, 20701, 100)}, 
        **{i: 'forest' for i in range(30100, 31201, 100)},
        **{i: 'arable' for i in range(40100, 40501, 100)},
        **{i: 'coastal' for i in range(50100, 51501, 100)}
    }
    default_value = np.nan  # Default value for codes not in the landuse_codes dictionary
    
    def recode_landuse(da):
        def recode_function(data_block):
            data_block = np.rint(data_block).astype(int)
    
            recoded = np.empty_like(data_block, dtype=object)
    
            # Apply the landuse mapping
            for key, value in landuse_codes.items():
                recoded[data_block == key] = value
    
            # Assign default value for unmatched keys
            recoded[~np.isin(data_block, list(landuse_codes.keys()))] = default_value
    
            return recoded
    
        recoded_data = xr.apply_ufunc(
            recode_function,
            da,
            input_core_dims=[[]],
            output_core_dims=[[]],
            vectorize=False,
            dask="parallelized",
            output_dtypes=[object],
        )
    
        return recoded_data
    
    # Load raster data using rioxarray with larger chunks
    original_raster_data = rxr.open_rasterio(file_path, masked=True, chunks=chunk_size)
    
    # Ensure the number of bands need to match the band names
    if len(band_names) != original_raster_data.sizes['band']:
        raise ValueError("The number of band names does not match the number of raster bands!")
    
    # Convert the raster data to xarray dataset and rename bands
    raster_dataset = original_raster_data.to_dataset(dim='band')
    raster_dataset = raster_dataset.rename({idx + 1: var for idx, var in enumerate(band_names)})
    
    # Apply preprocessing: recoding landuse type
    landuse_band = raster_dataset['landuse_type']
    recoded_landuse_band = recode_landuse(landuse_band)
    raster_dataset['landuse_type'] = recoded_landuse_band
    
    # Replace -9999 with NaN before stacking data
    raster_dataset = raster_dataset.where(raster_dataset != -9999, np.nan)
    
    # Process in tiles to avoid memory issues
    y_size, x_size = raster_dataset.y.size, raster_dataset.x.size
    tile_size = 20000 # Adjust tile size based on the available computing power
    
    # Create empty list to store processed chunks
    processed_chunks = []
    
    # Process in tiles
    for y_start in range(0, y_size, tile_size):
        y_end = min(y_start + tile_size, y_size)
        for x_start in range(0, x_size, tile_size):
            x_end = min(x_start + tile_size, x_size)
            
            # Extract tile
            tile = raster_dataset.isel(y=slice(y_start, y_end), x=slice(x_start, x_end))
            
            # Stack dimensions for this tile
            stacked_tile = tile.stack(dim_0=('y', 'x'))
            
            # Drop NaN values
            filtered_tile = stacked_tile.dropna(dim='dim_0', how='any')
                
            # Store the processed tile
            processed_chunks.append(filtered_tile)
    
    # If there is no valid data; return None
    if not processed_chunks:
        return None, original_raster_data
        
    # Combine all processed tiles into one single dataframe
    final_raster_dataset = xr.concat(processed_chunks, dim= 'dim_0')
    
    return final_raster_dataset, original_raster_data

#### Standardize and create dummies for data

In [ ]:
# Standardization of enviornmental covariates 
class CustomStandardScalerOptimized(TransformerMixin):
    def __init__(self, vars_to_standardize):
        self.vars_to_standardize = vars_to_standardize
        self.scaler = StandardScaler()
        
    def fit(self, dataset, y=None):
        # Check if we're working with a DataFrame (training) or xarray dataset (raster)
        if isinstance(dataset, pd.DataFrame):
            # For pandas DataFrame, we can use sklearn's StandardScaler directly
            # Extract features to standardize
            features = dataset[self.vars_to_standardize].values
            self.scaler.fit(features)
            return self
            
        # For xarray Dataset, use chunked computation to avoid memory issues
        means = []
        variances = []
        number_samples = 0
        
        for var in self.vars_to_standardize:
            # Process each variable's data in chunks
            data = dataset[var].data.flatten()
            
            # If data is a dask array:
            if isinstance(data, da.Array):
                # Use chunks for efficient memory management
                data = data.rechunk((min(data.shape[0] // 10, 1000000),))
                # Compute mean and variance for this variable
                chunk_mean = data.mean().compute()
                chunk_var = data.var().compute()
            else:
                chunk_mean = np.mean(data)
                chunk_var = np.var(data)
            
            number_chunk = data.size
            
            means.append(chunk_mean)
            variances.append(chunk_var)
            number_samples = number_chunk
        
        # Store statistics in the scaler
        self.scaler.mean_ = np.array(means)
        self.scaler.var_ = np.array(variances)
        self.scaler.scale_ = np.sqrt(self.scaler.var_)
        self.scaler.n_samples_seen_ = number_samples
        
        return self
    
    def transform(self, dataset):
         # Check if the type of dataset is either a DataFrame or xarray dataset; if the dataset is a pandas data frame then apply sklearn's standard scaler directly. 
        if isinstance(dataset, pd.DataFrame):
            transformed_values = self.scaler.transform(dataset[self.vars_to_standardize].values)
            result = dataset.copy()
            result[self.vars_to_standardize] = transformed_values
            return result
        
        # For xarray Dataset, transform each variable separately
        standardized = dataset.copy()
        
        # Process each variable separately to avoid memory issues
        for idx, var in enumerate(self.vars_to_standardize):
            data = dataset[var].data
            
            if isinstance(data, da.Array):
                chunk_size = min(data.shape[0] // 10, 10000)
            
                # If chunk_size is 0, set it to 1
                if chunk_size == 0:
                    chunk_size = 1 
    
                data = data.rechunk(chunks=(chunk_size, *data.shape[1:]))
                
                # Apply standardization formula
                std_data = (data - self.scaler.mean_[idx]) / self.scaler.scale_[idx]
                
                # Create a new data with the standardized data
                standardized[var] = xr.DataArray(
                    std_data, 
                    dims=dataset[var].dims, 
                    coords=dataset[var].coords
                )
            else:
                # Convert to dask array if it's not already
                dask_data = da.from_array(data, chunks=(min(data.shape[0] // 10, 1000000), *data.shape[1:]))
                std_data = (dask_data - self.scaler.mean_[idx]) / self.scaler.scale_[idx]
                
                standardized[var] = xr.DataArray(
                    std_data, 
                    dims=dataset[var].dims, 
                    coords=dataset[var].coords
                )
                
        return standardized

def convert_to_binary_columns(ds, variable_name, unique_categories):
    """
    Converts a categorical variable in an xarray Dataset into binary columns.
    """
    if variable_name not in ds:
        raise ValueError(f"The variable '{variable_name}' does not exist in the dataset!!")
        
    dummy_ds = xr.Dataset()
    
    for category in unique_categories:
        binary_variable_name = f"{variable_name}_{category}"
        dummy_ds[binary_variable_name] = xr.where(ds[variable_name] == category, 1, 0)

    ds = ds.drop_vars(variable_name)
    ds = xr.merge([ds, dummy_ds])

    return ds

# Export the file to tif
def rasterize_points_to_reference_grid(gdf, value_column, reference_raster_path, output_path, nodata=-9999):
    """
    Rasterise a point GeoDataFrame to match an existing reference raster grid
    
    Parameters:
    - gdf: GeoDataFrame with point geometries
    - value_column: Column name containing values to rasterize
    - reference_raster_path: Path to an existing raster to use as reference grid
    - output_path: Path to save the output GeoTIFF
    - nodata: Value to use for nodata pixels
    """
    # Open the reference raster to get its properties
    with rasterio.open(reference_raster_path) as src:
        meta = src.meta.copy()
        transform = src.transform
        height = src.height
        width = src.width
    
    # Determine output dtype based on the input column's dtype
    if np.issubdtype(gdf[value_column].dtype, np.floating):
        out_dtype = rasterio.float32
    elif np.issubdtype(gdf[value_column].dtype, np.integer):
        out_dtype = rasterio.int32
    else:
        out_dtype = rasterio.float32  # Default to float32
    
    # Create an empty raster matched to the reference
    raster = np.ones((height, width), dtype=out_dtype) * nodata
    
    # Convert points to shapes with values for rasterization
    shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf[value_column])]
    
    # Rasterize the points onto the reference grid
    rasterized = features.rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=nodata,
        all_touched=False,
        dtype=out_dtype  # Use the determined dtype
    )
    
    # Update metadata for the new raster
    meta.update({
        'count': 1,
        'nodata': nodata,
        'dtype': out_dtype
    })
    
    # Save the raster to a GeoTIFF
    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(rasterized, 1)
    
    print(f"GeoTIFF saved to {output_path}")

### Public access land

#### Read Clipped Estonia Raster Data: Public Access Land

In [ ]:
# Define band names for the raster
band_names = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type', 'dissimilarity_index']

# Path to the raster file
file_path = r"merged_public_land_candidate_area.tif"

preprocessed_raster, original_raster = preprocess_raster_data(file_path, band_names)

# Load raster data using rioxarray with larger chunks
original_raster_data = rxr.open_rasterio(file_path, masked = True, chunks = 1000)


# Convert the raster data to an xarray dataset and rename bands
raster_dataset = original_raster_data.to_dataset(dim = 'band')

raster_dataset = raster_dataset.where(raster_dataset != -9999, np.nan)

#### Computation

In [ ]:
training_columns =  ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']
drop_training_df = training_df[training_columns]

processed_raster_data = preprocessed_raster.copy()

# Create binary dummy variables for landuse type
dummy_training_data = convert_to_binary_columns(training_df, 'landuse_type', ['arable', 'forest', 'grassland', 'wetland'])
dummy_raster_data = convert_to_binary_columns(processed_raster_data, 'landuse_type', ['arable', 'forest', 'grassland', 'wetland'])

# Define the columns to standardise
cols_to_standardize = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained','landuse_type_arable', 'landuse_type_forest','landuse_type_grassland','landuse_type_wetland']
custom_scaler = CustomStandardScalerOptimized(cols_to_standardize)

# Fit training data to standardise features
dummy_scaled_training_data = custom_scaler.fit_transform(dummy_training_data)
dummy_scaled_testing_data = custom_scaler.transform(dummy_raster_data)

dummy_scaled_testing_data_df = dummy_scaled_testing_data.to_dataframe()
geometry = [Point(xy) for xy in zip(dummy_scaled_testing_data_df.index.get_level_values('x'), 
                                    dummy_scaled_testing_data_df.index.get_level_values('y'))]
gdf_test = gpd.GeoDataFrame(dummy_scaled_testing_data_df, geometry=geometry)
gdf_test.set_crs(epsg = 3301, inplace = True)
gdf_test = gdf_test.reset_index(drop = True)

# select the necessary columns
gdf_test = gdf_test[['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi',
       'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer',
       'bsi_autumn', 'drained', 'landuse_type_arable', 'landuse_type_forest', 'landuse_type_grassland',
       'landuse_type_wetland', 'dissimilarity_index', 'geometry']]

# Load SHAP values
shap_values = load(r"baseline_aoa_shap.pkl")

shap_columns = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 
                'bsi_autumn', 'drained', 'landuse_type_arable', 'landuse_type_forest', 'landuse_type_grassland', 'landuse_type_wetland']
gdf_test[shap_columns] = gdf_test[shap_columns] * shap_values

#### Kmeans Clustering Algorithm

In [ ]:
kmeans_df = gdf_test.copy()

# select the necessary columns for K-means
kmeans_cols = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 
                   'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 
                   'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type_arable', 
                   'landuse_type_forest', 'landuse_type_grassland', 'landuse_type_wetland']

kmeans_df = kmeans_df[kmeans_cols]

# Create k-means cluster
kmeans = KMeans(n_clusters = 200, init = 'k-means++', random_state = seed, n_init = 10, max_iter = 300)

kmeans.fit(kmeans_df)

# Add the cluster labels to the dataframe as a new column
cluster_labels = kmeans.labels_
gdf_test['cluster'] = cluster_labels

# Save it as tif format
rasterize_points_to_reference_grid(gdf = gdf_test, value_column = 'cluster', reference_raster_path = r"estonia_reference_grid.tif", output_path = f"kmeans_200_public_land.tif")